# Coding Agent From Scratch

Modern AI coding tools — Claude Code, Cursor, GitHub Copilot Chat — feel like magic until you look inside them. At their core is a straightforward loop: send a message to a language model, parse any tool calls from the response, execute those tools, feed the results back, and repeat until the task is done. The complexity lies in the details: how to stream responses token by token, how to represent dozens of tools in a way the model can call correctly, how to avoid running out of context window, how to keep the agent from getting stuck in a loop, how to ask for user approval before a destructive command, and how to coordinate multiple specialized sub-agents on complex multi-part tasks.

This series builds a complete async coding agent from the lowest layer upward — starting with an OpenAI-compatible streaming client and finishing with a Flet desktop application. Every module is written for clarity and replaceability: you can swap the LLM provider, add tools, change the approval policy, or plug in a different UI without touching the rest of the stack.

## About This Series

**Goal.** Build a [production-quality AI coding agent]{.mark} — similar in architecture to Claude Code — from first principles. The agent code lives in `src/notebooks/agent/` as an importable async Python package, and every notebook in the series adds one layer of the stack.

**Audience.** Someone comfortable with Python async/await and the OpenAI API. Familiarity with Pydantic and Flet is helpful but not required — those libraries are introduced as needed.

**Stack.** [OpenRouter](https://openrouter.ai) as the LLM gateway (default model: `anthropic/claude-sonnet-4`), [Pydantic](https://docs.pydantic.dev) for configuration and tool schemas, [Flet](https://flet.dev) for the desktop UI. All secrets come from environment variables (`OPENROUTER_API_KEY`). The package is installed in editable mode via `uv sync` and any cell can `import notebooks.agent` immediately.

## Course Notebooks

| # | Title | What We Build | Key Concepts |
|---|---|---|---|
| 01 | [Streaming LLM Client](/notebooks/apps/cda/01-client.html) | `LLMClient`: async streaming wrapper for OpenAI-compatible APIs | Streaming deltas, tool-call accumulation, `StreamEvent` types, token usage |
| 02 | [The Tool System](/notebooks/apps/cda/02-tools.html) | 7 builtin tools + `ToolRegistry` + Pydantic schemas | `Tool` ABC, `ToolKind`, `ToolResult`, `ToolRegistry.invoke`, OpenAI function-calling format |
| 03 | [The Agent Loop](/notebooks/apps/cda/03-agent.html) | `Session` + `Agent`: the full multi-turn agentic loop | System prompt assembly, message history, `AgentEvent` stream, turn-by-turn execution |
| 04 | [Hardening the Agent](/notebooks/apps/cda/04-hardening.html) | `ContextManager`, `ChatCompactor`, `LoopDetector`, `ApprovalManager`, `SubAgentTool` | Context pruning, LLM-based summarization, loop detection, approval policies, sub-agent orchestration |
| 05 | [GUI 1: Chat Interface](/notebooks/apps/cda/05-ui.html) | Flet chat application wrapping the agent | `MessageBubble`, `ToolCallCard`, `StatusBar`, `AgentApp` event loop integration |
| 06 | [GUI 2: Tools & Persistence](/notebooks/apps/cda/06-ui2.html) | 4 new builtin tools, session save/load, slash commands | `FetchUrlTool`, `WebSearchTool`, `MemoryTool`, `TodoTool`, `Session.save/load`, `handle_command`, `CommandResult` |
| 07 | [GUI 3: MCP Integration](/notebooks/apps/cda/07-ui3.html) | Connect MCP servers to the agent and UI | `MCPServerConfig`, `MCPToolAdapter`, `MCPManager`, stdio & HTTP transports, `/mcp` command, purple tool cards |

: {tbl-colwidths="[5,20,35,40]"}

## What You'll Build Understanding Of

- How streaming LLM responses work at the protocol level — deltas, finish reasons, tool-call chunks
- The function-calling format expected by OpenAI-compatible APIs and how to produce JSON schemas from Pydantic models
- The agentic loop: turn counting, multi-tool execution, message accumulation
- Context window management: token estimation, mechanical pruning, and LLM-based compaction
- Loop detection: signature hashing, repeat counting, cycle detection
- Approval policies: YOLO, AUTO, ON_REQUEST, NEVER — and how to implement a simple safety gate
- Sub-agent orchestration: spawning a focused child agent with a restricted tool set to handle a scoped sub-task
- Integrating an async agent into a Flet desktop UI with in-place streaming updates

## Prerequisites

- **Python 3.13+** with `uv` installed (`make venv` to create the environment)
- **Async Python:** comfortable reading `async def` / `async for` / `await`
- **OpenAI API familiarity:** you have called a chat-completion endpoint before
- **Pydantic basics:** `BaseModel`, `Field`, validation — introduced briefly but not taught from scratch
- **Environment:** `OPENROUTER_API_KEY` set in your shell for the live demos in NB01–NB04; NB05 needs this to run the agent

## How to Read This Series

**Linear read.** Each notebook builds on the previous one's source files. Reading in order (01 → 05) gives you the full architectural story.

**Jump in.** If you only care about context management or loop detection, NB04 is self-contained once you understand that `Session` holds the message list (introduced in NB03).

**Use the package.** After `uv sync`, every module is importable: `from notebooks.agent import Agent, Config`. The quickest way to run the agent is:

```python
from notebooks.agent import Agent
agent = Agent()
async for event in agent.run("List the Python files in this directory"):
    print(event)
```

**Run the desktop app.** After reading NB05:

```bash
uv run flet run src/notebooks/agent/ui/app.py
```